# 02 MBA Rules and Recommendations

Compute interpretable two-item market basket rules directly in pandas and overlay customer outcomes and available margin coverage.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "EDA" / "outputs").exists() and (candidate / "product_mba").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA/outputs and product_mba")

PROJECT_ROOT = find_project_root()
EDA_OUTPUTS = PROJECT_ROOT / "EDA" / "outputs_finals"  # FINALS cohort
MBA_DIR = PROJECT_ROOT / "product_mba"
MBA_OUTPUTS = MBA_DIR / "outputs"
MBA_OUTPUTS.mkdir(exist_ok=True)

print("Project root found")
print("EDA outputs: EDA/outputs_finals (finals cohort)")
print("MBA outputs: product_mba/outputs")


Project root found
EDA outputs: EDA/outputs_finals (finals cohort)
MBA outputs: product_mba/outputs


In [2]:
import math
import numpy as np
import pandas as pd

customers = pd.read_parquet(EDA_OUTPUTS / "customers.parquet")
try:
    margin_lines = pd.read_parquet(MBA_OUTPUTS / "margin_line_enriched.parquet")
except FileNotFoundError:
    margin_lines = pd.DataFrame()

thresholds = {
    "category": {"min_support": 0.005, "min_confidence": 0.10, "min_lift": 1.10, "min_co_orders": 1, "min_item_orders": 1},
    "handle": {"min_support": 0.003, "min_confidence": 0.08, "min_lift": 1.20, "min_co_orders": 1, "min_item_orders": 1},
    "sku_flavor": {"min_support": 0.0, "min_confidence": 0.0, "min_lift": 1.30, "min_co_orders": 10, "min_item_orders": 50},
}

def build_pair_rules(item_df, item_col, level):
    item_df = item_df[["order_id", "customer_id", item_col]].dropna().drop_duplicates(["order_id", item_col]).copy()
    order_count = item_df["order_id"].nunique()
    item_orders = item_df.groupby(item_col)["order_id"].nunique().rename("antecedent_orders")
    basket_items = item_df.groupby("order_id")[item_col].apply(lambda s: sorted(set(s))).reset_index(name="items")

    pair_rows = []
    for order_id, items in zip(basket_items["order_id"], basket_items["items"]):
        if len(items) < 2:
            continue
        for i, a in enumerate(items):
            for b in items[i + 1:]:
                pair_rows.append((a, b, order_id))
    if not pair_rows:
        return pd.DataFrame()

    pairs = pd.DataFrame(pair_rows, columns=["item_a", "item_b", "order_id"])
    pair_counts = pairs.groupby(["item_a", "item_b"])["order_id"].nunique().reset_index(name="co_orders")

    directional = pd.concat([
        pair_counts.rename(columns={"item_a": "antecedent", "item_b": "consequent"}),
        pair_counts.rename(columns={"item_b": "antecedent", "item_a": "consequent"}),
    ], ignore_index=True)
    directional = directional.merge(item_orders.reset_index().rename(columns={item_col: "antecedent"}), on="antecedent", how="left")
    directional = directional.merge(
        item_orders.reset_index().rename(columns={item_col: "consequent", "antecedent_orders": "consequent_orders"}),
        on="consequent", how="left",
    )
    directional["total_orders"] = order_count
    directional["support"] = directional["co_orders"] / order_count
    directional["antecedent_support"] = directional["antecedent_orders"] / order_count
    directional["consequent_support"] = directional["consequent_orders"] / order_count
    directional["confidence"] = directional["co_orders"] / directional["antecedent_orders"]
    directional["lift"] = directional["confidence"] / directional["consequent_support"]
    directional["leverage"] = directional["support"] - (directional["antecedent_support"] * directional["consequent_support"])
    directional["conviction"] = np.where(
        directional["confidence"] < 1,
        (1 - directional["consequent_support"]) / (1 - directional["confidence"]),
        np.inf,
    )
    directional["level"] = level

    t = thresholds[level]
    filtered = directional[
        (directional["support"] >= t["min_support"])
        & (directional["confidence"] >= t["min_confidence"])
        & (directional["lift"] >= t["min_lift"])
        & (directional["co_orders"] >= t["min_co_orders"])
        & (directional["antecedent_orders"] >= t["min_item_orders"])
        & (directional["consequent_orders"] >= t["min_item_orders"])
    ].copy()
    return filtered.sort_values(["lift", "co_orders", "confidence"], ascending=[False, False, False])

def add_outcomes(rules, item_df, item_col):
    if rules.empty:
        return rules
    cust = customers.copy()
    cust["customer_id"] = cust["customer_id"].astype(str)
    item_df = item_df[["customer_id", item_col]].dropna().drop_duplicates()
    item_df["customer_id"] = item_df["customer_id"].astype(str)

    rows = []
    for idx, row in rules.iterrows():
        customers_with_both = item_df[item_df[item_col].isin([row["antecedent"], row["consequent"]])].groupby("customer_id")[item_col].nunique()
        both_ids = customers_with_both[customers_with_both >= 2].index
        segment = cust[cust["customer_id"].isin(both_ids)]
        rows.append({
            "rule_index": idx,
            "both_item_customers": len(segment),
            "both_item_repeat_rate": segment["is_repeat"].mean() if len(segment) else np.nan,
            "both_item_avg_ltv": segment["total_revenue"].mean() if len(segment) else np.nan,
            "both_item_pct_subscribed": segment["ever_subscribed"].mean() if len(segment) else np.nan,
        })
    outcome = pd.DataFrame(rows).set_index("rule_index")
    return rules.join(outcome)

def add_margin_overlay(rules, level):
    if rules.empty or margin_lines.empty:
        rules["margin_overlay_available"] = False
        return rules
    if level == "category":
        item_col = "product_category"
    elif level == "handle":
        item_col = "Line: Product Handle"
    else:
        item_col = "sku_norm"
    if item_col not in margin_lines.columns:
        rules["margin_overlay_available"] = False
        return rules
    m = (
        margin_lines.dropna(subset=[item_col])
        .groupby(item_col)
        .agg(
            item_revenue_sgd=("net_revenue", "sum"),
            item_covered_revenue_sgd=("net_revenue", lambda s: s[margin_lines.loc[s.index, "has_high_confidence_cost"]].sum()),
            item_est_gross_profit_sgd=("estimated_gross_profit", "sum"),
        )
        .reset_index()
    )
    m["item_margin_coverage_pct"] = np.where(m["item_revenue_sgd"] > 0, m["item_covered_revenue_sgd"] / m["item_revenue_sgd"], np.nan)
    m["item_est_margin_pct"] = np.where(m["item_covered_revenue_sgd"] > 0, m["item_est_gross_profit_sgd"] / m["item_covered_revenue_sgd"], np.nan)
    ant = m.add_prefix("antecedent_").rename(columns={f"antecedent_{item_col}": "antecedent"})
    con = m.add_prefix("consequent_").rename(columns={f"consequent_{item_col}": "consequent"})
    rules = rules.merge(ant, on="antecedent", how="left").merge(con, on="consequent", how="left")
    rules["margin_overlay_available"] = rules[["antecedent_item_margin_coverage_pct", "consequent_item_margin_coverage_pct"]].notna().any(axis=1)
    return rules


In [3]:
rule_tables = {}
for level in ["category", "handle", "sku_flavor"]:
    item_df = pd.read_parquet(MBA_OUTPUTS / f"basket_items_{level}.parquet")
    item_col = item_df.columns[-1]
    rules = build_pair_rules(item_df, item_col, level)
    rules = add_outcomes(rules, item_df, item_col)
    rules = add_margin_overlay(rules, level)
    rule_tables[level] = rules
    rules.to_csv(MBA_OUTPUTS / f"mba_rules_{level}.csv", index=False)
    print(level, "rules:", len(rules))
    display(rules.head(10))

recommendations = []
for level, rules in rule_tables.items():
    if rules.empty:
        continue
    top = rules.sort_values(["lift", "co_orders", "both_item_avg_ltv"], ascending=[False, False, False]).head(15).copy()
    for _, row in top.iterrows():
        recommendations.append({
            "level": level,
            "antecedent": row["antecedent"],
            "consequent": row["consequent"],
            "recommendation_type": "Cross-sell / bundle candidate",
            "rationale": f"{int(row['co_orders'])} co-orders, lift {row['lift']:.2f}, confidence {row['confidence']:.1%}",
            "support": row["support"],
            "confidence": row["confidence"],
            "lift": row["lift"],
            "co_orders": row["co_orders"],
            "avg_ltv_for_both_item_customers": row.get("both_item_avg_ltv", np.nan),
            "repeat_rate_for_both_item_customers": row.get("both_item_repeat_rate", np.nan),
            "margin_overlay_available": row.get("margin_overlay_available", False),
        })
recommendations_df = pd.DataFrame(recommendations)
recommendations_df.to_csv(MBA_OUTPUTS / "mba_recommendations.csv", index=False)
print("Saved recommendations:", len(recommendations_df))
display(recommendations_df.head(20))


category rules: 4


,antecedent,consequent,co_orders,antecedent_orders,consequent_orders,total_orders,support,antecedent_support,consequent_support,confidence,...,antecedent_item_covered_revenue_sgd,antecedent_item_est_gross_profit_sgd,antecedent_item_margin_coverage_pct,antecedent_item_est_margin_pct,consequent_item_revenue_sgd,consequent_item_covered_revenue_sgd,consequent_item_est_gross_profit_sgd,consequent_item_margin_coverage_pct,consequent_item_est_margin_pct,margin_overlay_available
0,Lean Protein,Accessories,382,1939,1068,7347,0.051994,0.263917,0.145365,0.197009,...,112726.027273,83009.317273,0.841370,0.736381,10934.647273,10607.271515,8929.311515,0.970061,0.841810,True
1,Accessories,Lean Protein,382,1068,1939,7347,0.051994,0.145365,0.263917,0.357678,...,10607.271515,8929.311515,0.970061,0.841810,133979.167273,112726.027273,83009.317273,0.841370,0.736381,True
2,Accessories,Clear Protein,341,1068,2130,7347,0.046414,0.145365,0.289914,0.319288,...,10607.271515,8929.311515,0.970061,0.841810,176994.175152,159600.916970,117771.706970,0.901730,0.737914,True
3,Clear Protein,Accessories,341,2130,1068,7347,0.046414,0.289914,0.145365,0.160094,...,159600.916970,117771.706970,0.901730,0.737914,10934.647273,10607.271515,8929.311515,0.970061,0.841810,True


handle rules: 12


,antecedent,consequent,co_orders,antecedent_orders,consequent_orders,total_orders,support,antecedent_support,consequent_support,confidence,...,antecedent_item_covered_revenue_sgd,antecedent_item_est_gross_profit_sgd,antecedent_item_margin_coverage_pct,antecedent_item_est_margin_pct,consequent_item_revenue_sgd,consequent_item_covered_revenue_sgd,consequent_item_est_gross_profit_sgd,consequent_item_margin_coverage_pct,consequent_item_est_margin_pct,margin_overlay_available
0,pureburn-fat-burner-capsules,green-tea-extract-capsules,29,125,73,7347,0.003947,0.017014,0.009936,0.232000,...,6006.836364,5127.986364,0.840569,0.853692,1752.195758,1436.594848,1052.814848,0.819883,0.732854,True
1,green-tea-extract-capsules,pureburn-fat-burner-capsules,29,73,125,7347,0.003947,0.009936,0.017014,0.397260,...,1436.594848,1052.814848,0.819883,0.732854,7146.157576,6006.836364,5127.986364,0.840569,0.853692,True
2,clear-protein-25g-single-sachet,lushprotein-lean-protein-40g-single-serve,101,292,322,7347,0.013747,0.039744,0.043827,0.345890,...,6600.480000,5391.930000,0.981058,0.816900,7471.640000,7276.740000,6209.360000,0.973915,0.853316,True
3,lushprotein-lean-protein-40g-single-serve,clear-protein-25g-single-sachet,101,322,292,7347,0.013747,0.043827,0.039744,0.313665,...,7276.740000,6209.360000,0.973915,0.853316,6727.920000,6600.480000,5391.930000,0.981058,0.816900,True
4,lean-protein-peach-oolong-pre-order,lushprotein-clear-shaker,30,85,1068,7347,0.004083,0.011569,0.145365,0.352941,...,7758.480000,5411.030000,1.000000,0.697434,10934.647273,10607.271515,8929.311515,0.970061,0.841810,True
5,lushprotein-lean-protein-40g-single-serve,lushprotein-clear-shaker,91,322,1068,7347,0.012386,0.043827,0.145365,0.282609,...,7276.740000,6209.360000,0.973915,0.853316,10934.647273,10607.271515,8929.311515,0.970061,0.841810,True
6,lushprotein-clear-shaker,lushprotein-lean-protein-40g-single-serve,91,1068,322,7347,0.012386,0.145365,0.043827,0.085206,...,10607.271515,8929.311515,0.970061,0.841810,7471.640000,7276.740000,6209.360000,0.973915,0.853316,True
7,clear-protein-25g-single-sachet,lushprotein-clear-shaker,72,292,1068,7347,0.009800,0.039744,0.145365,0.246575,...,6600.480000,5391.930000,0.981058,0.816900,10934.647273,10607.271515,8929.311515,0.970061,0.841810,True
8,green-tea-extract-capsules,lean-protein,25,73,1582,7347,0.003403,0.009936,0.215326,0.342466,...,1436.594848,1052.814848,0.819883,0.732854,118749.047273,97690.807273,71388.927273,0.822666,0.730764,True
9,discovery-sampler,lushprotein-clear-shaker,40,189,1068,7347,0.005444,0.025725,0.145365,0.211640,...,5625.080000,4783.440000,1.000000,0.850377,10934.647273,10607.271515,8929.311515,0.970061,0.841810,True


sku_flavor rules: 180


,antecedent,consequent,co_orders,antecedent_orders,consequent_orders,total_orders,support,antecedent_support,consequent_support,confidence,...,antecedent_item_covered_revenue_sgd,antecedent_item_est_gross_profit_sgd,antecedent_item_margin_coverage_pct,antecedent_item_est_margin_pct,consequent_item_revenue_sgd,consequent_item_covered_revenue_sgd,consequent_item_est_gross_profit_sgd,consequent_item_margin_coverage_pct,consequent_item_est_margin_pct,margin_overlay_available
0,0724999810463 | Lean Protein Single Serve | 1 ...,0724999810470 | Lean Protein Single Serve | 1 ...,38,51,51,7564,0.005024,0.006742,0.006742,0.745098,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,0724999810470 | Lean Protein Single Serve | 1 ...,0724999810463 | Lean Protein Single Serve | 1 ...,38,51,51,7564,0.005024,0.006742,0.006742,0.745098,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,0724999810487 | CLEAR PROTEIN | 25g Sachet (1 ...,0724999810494 | CLEAR PROTEIN | 25g Sachet (1 ...,60,71,71,7564,0.007932,0.009387,0.009387,0.845070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,0724999810494 | CLEAR PROTEIN | 25g Sachet (1 ...,0724999810487 | CLEAR PROTEIN | 25g Sachet (1 ...,60,71,71,7564,0.007932,0.009387,0.009387,0.845070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,LEAN-THA-40G-V1 | Lean Protein Single Serve | ...,LEAN-TAR-40G-V1 | Lean Protein Single Serve | ...,56,91,74,7564,0.007403,0.012031,0.009783,0.615385,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
5,LEAN-TAR-40G-V1 | Lean Protein Single Serve | ...,LEAN-THA-40G-V1 | Lean Protein Single Serve | ...,56,74,91,7564,0.007403,0.009783,0.012031,0.756757,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
6,0724999810470 | LEAN PROTEIN | 40g Sachet (1 s...,0724999810463 | LEAN PROTEIN | 40g Sachet (1 s...,116,125,131,7564,0.015336,0.016526,0.017319,0.928000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
7,0724999810463 | LEAN PROTEIN | 40g Sachet (1 s...,0724999810470 | LEAN PROTEIN | 40g Sachet (1 s...,116,131,125,7564,0.015336,0.017319,0.016526,0.885496,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
8,CLEAR-GRA-25G-V2 | Clear Protein Single Serve ...,CLEAR-PEA-25G-V2 | Clear Protein Single Serve ...,66,86,110,7564,0.008726,0.011370,0.014543,0.767442,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
9,CLEAR-PEA-25G-V2 | Clear Protein Single Serve ...,CLEAR-GRA-25G-V2 | Clear Protein Single Serve ...,66,110,86,7564,0.008726,0.014543,0.011370,0.600000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


Saved recommendations: 31


,level,antecedent,consequent,recommendation_type,rationale,support,confidence,lift,co_orders,avg_ltv_for_both_item_customers,repeat_rate_for_both_item_customers,margin_overlay_available
0,category,Lean Protein,Accessories,Cross-sell / bundle candidate,"382 co-orders, lift 1.36, confidence 19.7%",0.051994,0.197009,1.355265,382,250.442432,0.409314,True
1,category,Accessories,Lean Protein,Cross-sell / bundle candidate,"382 co-orders, lift 1.36, confidence 35.8%",0.051994,0.357678,1.355265,382,250.442432,0.409314,True
2,category,Accessories,Clear Protein,Cross-sell / bundle candidate,"341 co-orders, lift 1.10, confidence 31.9%",0.046414,0.319288,1.101320,341,310.202788,0.456693,True
3,category,Clear Protein,Accessories,Cross-sell / bundle candidate,"341 co-orders, lift 1.10, confidence 16.0%",0.046414,0.160094,1.101320,341,310.202788,0.456693,True
4,handle,pureburn-fat-burner-capsules,green-tea-extract-capsules,Cross-sell / bundle candidate,"29 co-orders, lift 23.35, confidence 23.2%",0.003947,0.232000,23.349370,29,1700.235303,0.600000,True
5,handle,green-tea-extract-capsules,pureburn-fat-burner-capsules,Cross-sell / bundle candidate,"29 co-orders, lift 23.35, confidence 39.7%",0.003947,0.397260,23.349370,29,1700.235303,0.600000,True
6,handle,clear-protein-25g-single-sachet,lushprotein-lean-protein-40g-single-serve,Cross-sell / bundle candidate,"101 co-orders, lift 7.89, confidence 34.6%",0.013747,0.345890,7.892102,101,549.949394,0.343434,True
7,handle,lushprotein-lean-protein-40g-single-serve,clear-protein-25g-single-sachet,Cross-sell / bundle candidate,"101 co-orders, lift 7.89, confidence 31.4%",0.013747,0.313665,7.892102,101,549.949394,0.343434,True
8,handle,lean-protein-peach-oolong-pre-order,lushprotein-clear-shaker,Cross-sell / bundle candidate,"30 co-orders, lift 2.43, confidence 35.3%",0.004083,0.352941,2.427958,30,228.129945,0.568182,True
9,handle,lushprotein-lean-protein-40g-single-serve,lushprotein-clear-shaker,Cross-sell / bundle candidate,"91 co-orders, lift 1.94, confidence 28.3%",0.012386,0.282609,1.944126,91,589.419678,0.443299,True


## What The MBA Output Tells Us

The market basket analysis produces three rule tables: category-level rules, product-handle rules, and SKU/flavor rules. A rule like `Accessories -> Lean Protein` means that among orders containing Accessories, Lean Protein appears more often than expected by chance. The most important columns are:

- `co_orders`: how many baskets contained both products.
- `confidence`: of baskets with the antecedent, the share that also contained the consequent.
- `lift`: how much more often the pair occurs together versus random chance. Values above `1.0` indicate positive association.
- `avg_ltv_for_both_item_customers` and `repeat_rate_for_both_item_customers`: whether customers who bought both items look commercially valuable.
- `margin_overlay_available`: whether cost coverage exists for margin-informed decisions. Treat margin as directional because `00_margin_data.ipynb` rated the margin data **usable with caveats**.

### Main Readout

1. Accessories are the clearest category-level cross-sell lever.
   Accessories pair with both Lean Protein and Clear Protein at meaningful scale: `Accessories -> Lean Protein` has 774 co-orders, 34.0% confidence, and 1.36 lift; `Accessories -> Clear Protein` has 734 co-orders, 32.3% confidence, and 1.30 lift.

2. Single-serve sachets behave like variety/discovery products.
   The strongest SKU/flavor rules are mostly Clear Protein Peach with White Grape, and Lean Protein Taro with Thai Milk Tea. These have very high lift because customers often buy these sachets together as flavor trials or bundles.

3. A few supplement pairings are small but very strong.
   Examples include Green Tea Extract with Pureburn Fat Burner, and Multivitamin Vegan with Super Omega-3. These have high lift but lower co-order volume, so they are better treated as targeted bundle tests rather than broad homepage strategy.

4. The recommendations table is not saying every high-lift pair should become a major campaign.
   Prioritize pairs with enough volume (`co_orders`), sensible confidence, and strategic fit. Very high lift with low volume can be useful for niche bundles, but not necessarily for a core growth bet.

### What To Do Exactly

1. Add cart/checkout cross-sell prompts for protein plus accessories.
   When a cart contains Lean Protein or Clear Protein, recommend shakers or relevant accessories. When a cart contains accessories only, recommend Lean Protein first, then Clear Protein.

2. Create or promote variety bundles for sachets.
   Bundle Clear Protein Peach + White Grape, and Lean Protein Taro + Thai Milk Tea. Position these as trial packs, discovery sets, or flavor-comparison bundles.

3. Use targeted supplement bundles, not broad campaigns.
   Test Green Tea Extract + Pureburn Fat Burner, Multivitamin Vegan + Super Omega-3, and Super Omega-3 + Creatine with small placements, email segments, or post-purchase offers.

4. Use `product_mba/outputs/mba_recommendations.csv` as the action list.
   Start with category and handle rows because they are easier to translate into merchandising. Use SKU/flavor rows mainly for flavor bundles and specific product-page recommendations.

5. Do not optimize purely on estimated margin yet.
   Margin coverage is strong for core hero categories but incomplete overall. Use the margin overlay as a guardrail, and ask LushProtein for updated SKU-level COGS/currency/effective-date confirmation before making margin-maximizing decisions.

### Recommended First Experiments

- Experiment 1: Protein + shaker/accessory checkout cross-sell.
- Experiment 2: Clear Protein Peach + White Grape sachet bundle.
- Experiment 3: Lean Protein Taro + Thai Milk Tea sachet bundle.
- Experiment 4: Small targeted supplement bundle test for Green Tea Extract + Pureburn Fat Burner.
- Experiment 5: Post-purchase recommendation email based on first purchased product family.

---
### Final metrics & scores  ·  *finals cohort*

**Rules retained after thresholds:** category **4** · handle **12** · SKU/flavor **180** (8,955 orders, finals SKU-analysis pool).

| Level | Rule | Co-orders | Confidence | Lift |
|---|---|--:|--:|--:|
| Category | Accessories ↔ Lean Protein | 382 | 35.8% / 19.7% | 1.36 |
| Category | Accessories ↔ Clear Protein | 341 | 31.9% / 16.0% | 1.10 |
| Handle | clear-shaker ↔ lean-protein | 277 | 25.9% / 17.5% | 1.20 |
| Handle | clear 25g sachet ↔ lean 40g single-serve | 101 | 34.6% | 7.89 |
| SKU/flavor | Clear Peach ↔ Clear White Grape (500g) | 168 | 26–40% | 4.58 |
| SKU/flavor | Lean Thai Milk Tea sachet ↔ LP Shaker | 125 | 92% | 14.7 |

**Read:** highest-*volume* cross-category lever = **Accessories / shaker ↔ core protein**; highest-*lift* = **flavor-variety & trial-pack pairs** (discovery behaviour). Both feed the retention re-scoring in `03`.
